<a href="https://colab.research.google.com/github/impos0108/Tutorials/blob/main/260608_Gangwon_wildfire_risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌲 강원도 산불 이력 기반 수종 분류 & 산불 위험도 분석

## 전체 파이프라인
```
① 산불 이력 데이터 수집 (GEE FIRMS + 산림청 통계)
        ↓
② 수종별 산불 발생 빈도 계산 (임상도 × 산불 이력)
        ↓
③ Sentinel-2 시계열로 수종 분류 딥러닝 (U-Net)
        ↓
④ 산불 위험도 지도 생성 (수종 × 경사도 × 건조도 × 산불이력)
        ↓
⑤ 시각화 & 위험도 리포트
```

| 항목 | 내용 |
|------|------|
| **분석 지역** | 강원도 전체 |
| **수종 클래스** | 침엽수 / 활엽수 / 혼효림 / 비산림 |
| **산불 이력** | FIRMS MODIS/VIIRS (2000~현재) |
| **위성** | Sentinel-2 (10m) |
| **런타임** | T4 GPU 권장 |

> ⚠️ **먼저**: `런타임 → 런타임 유형 변경 → T4 GPU` 설정

## Section 0 — 패키지 설치 & 인증

In [ ]:
!pip install -q geedim

In [ ]:
%%capture
!pip install -q geemap segmentation-models-pytorch
print('설치 완료')

In [ ]:
import ee, geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import requests, io, os, warnings
from PIL import Image as PILImage
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import segmentation_models_pytorch as smp
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# GEE 인증
import getpass
PROJECT_ID = input('GEE Project ID를 입력하세요: ').strip()
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print(f'GEE initialized: {PROJECT_ID}')

## Section 1 — 연구 지역 & 공통 함수 설정

In [ ]:
# ── 강원도 ROI ────────────────────────────────────────────────
# 강원도 전체 경계 (GEE 행정경계 데이터)
# ── 강원도 ROI 수정 ───────────────────────────────────────────
GANGWON = (
    ee.FeatureCollection('FAO/GAUL/2015/level1')
    .filter(ee.Filter.eq('ADM1_NAME', 'Kang-won-do'))  # ← 수정
    .geometry()
)

# 유효성 확인
area = GANGWON.area().getInfo()
print(f'Gangwon-do area: {area/1e6:.0f} km²')  # 약 16,000 km² 나오면 정상

# 분석 기간
S2_START = '2022-05-01'   # Sentinel-2 수종 분류용
S2_END   = '2022-10-31'   # 여름철: 수종별 스펙트럴 차이 최대

FIRMS_START = '2012-01-01'  # FIRMS 산불 이력 시작
FIRMS_END   = '2024-12-31'  # FIRMS 산불 이력 종료

# 수종 클래스 정의
SPECIES_CLASSES = {
    0: 'Coniferous',    # 침엽수 (소나무, 잣나무, 낙엽송)
    1: 'Broadleaf',     # 활엽수 (참나무류, 신갈나무)
    2: 'Mixed',         # 혼효림
    3: 'Non-forest',    # 비산림 (초지, 농경지, 나지, 수계)
}
SPECIES_COLORS = ['#e74c3c', '#27ae60', '#f39c12', '#95a5a6']

# 수종별 산불 위험 기본 계수 (문헌 기반)
# 침엽수(소나무)=1.0 기준 상대값
SPECIES_RISK = {
    'Coniferous': 1.00,   # 송진 함량 높음 → 인화성 최강
    'Mixed':      0.60,   # 침엽/활엽 혼합
    'Broadleaf':  0.25,   # 수분 함량 높음
    'Non-forest': 0.05,   # 산불 확산 차단
}

CENTER = [37.5, 128.2]  # 강원도 중심

print('Study area: Gangwon-do, South Korea')
print(f'Sentinel-2 period: {S2_START} ~ {S2_END}')
print(f'FIRMS fire history: {FIRMS_START} ~ {FIRMS_END}')
print(f'Species classes: {list(SPECIES_CLASSES.values())}')

In [ ]:
# ── 공통 시각화 함수 ──────────────────────────────────────────

def get_thumb_image(image, bands, min_val, max_val, gamma=1.4, size=512):
    """GEE 이미지 → NumPy RGB 배열"""
    url = image.getThumbURL({
        'dimensions': size, 'region': GANGWON,
        'bands': bands, 'min': min_val, 'max': max_val,
        'gamma': gamma, 'format': 'png'
    })
    resp = requests.get(url, timeout=120)
    return np.array(PILImage.open(io.BytesIO(resp.content)))


def get_index_image(image, min_val, max_val, palette, size=512):
    """단밴드 GEE 이미지 → 컬러맵 NumPy 배열"""
    url = image.getThumbURL({
        'dimensions': size, 'region': GANGWON,
        'min': min_val, 'max': max_val,
        'palette': palette, 'format': 'png'
    })
    resp = requests.get(url, timeout=120)
    if resp.status_code != 200 or 'image' not in resp.headers.get('Content-Type',''):
        raise ValueError(f'GEE error: {resp.text[:500]}')
    return np.array(PILImage.open(io.BytesIO(resp.content)))


print('Helper functions ready')

## Section 2 — 산불 이력 데이터 수집 & 수종별 빈도 계산

**FIRMS (Fire Information for Resource Management System)**
- NASA가 운영하는 전 세계 위성 화점(Fire Hotspot) 데이터
- MODIS: 2000년~현재, 1km 해상도
- VIIRS: 2012년~현재, 375m 해상도 (더 정밀)
- GEE에서 무료 접근 가능

In [ ]:
# ── 강원도 이름 확인 ──────────────────────────────────────────
gaul = ee.FeatureCollection('FAO/GAUL/2015/level1')

# 한국 행정구역 이름 전체 출력
korea = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Republic of Korea'))
names = korea.aggregate_array('ADM1_NAME').getInfo()
print('[한국 행정구역 이름 목록]')
for n in names:
    print(f'  {n}')

In [ ]:
# ── FIRMS VIIRS 화점 데이터 로드 ──────────────────────────────
print('Loading FIRMS fire hotspot data...')

firms_viirs = (
    ee.ImageCollection('FIRMS')
    .filterBounds(GANGWON)
    .filterDate(FIRMS_START, FIRMS_END)
    .select('T21')   # 21μm 밝기온도 (화점 신뢰도)
)

# 화점 밀도 맵: 각 픽셀에서 화점이 감지된 횟수 합산
fire_count = (
    firms_viirs
    .map(lambda img: img.gt(300).toInt())  # 밝기온도 300K 이상 = 화점
    .sum()
    .clip(GANGWON)
    .rename('fire_count')
)

# 전체 화점 픽셀 수 확인
total_fires = fire_count.gt(0).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=GANGWON, scale=500,
    maxPixels=1e12, bestEffort=True
).getInfo()

print(f'FIRMS period: {FIRMS_START} ~ {FIRMS_END}')
print(f'Total fire pixels detected: {total_fires.get("fire_count", 0):,}')

# 화점 밀도 시각화
print('\nDownloading fire density map...')
fire_arr = get_index_image(
    fire_count, 0, 10,
    ['ffffff','ffffb2','fecc5c','fd8d3c','f03b20','bd0026']
)
rgb_arr = get_thumb_image(
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(GANGWON)
    .filterDate('2022-07-01','2022-09-30')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median().clip(GANGWON),
    ['B4','B3','B2'], 0, 3000
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    f'Gangwon-do Fire Hotspot Density\n'
    f'FIRMS VIIRS ({FIRMS_START[:4]}–{FIRMS_END[:4]})',
    fontsize=13, fontweight='bold'
)
axes[0].imshow(rgb_arr)
axes[0].set_title('Sentinel-2 RGB (Summer 2022)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(fire_arr)
axes[1].set_title('Fire Hotspot Count (FIRMS)', fontsize=11)
axes[1].axis('off')

sm = plt.cm.ScalarMappable(
    cmap=plt.cm.YlOrRd,
    norm=mcolors.Normalize(vmin=0, vmax=10)
)
sm.set_array([])
plt.colorbar(sm, ax=axes[1], fraction=0.046, pad=0.04,
             label='Fire detection count')

plt.tight_layout()
plt.savefig('fire_hotspot_density.png', bbox_inches='tight', dpi=150)
plt.show()
print("'fire_hotspot_density.png' saved.")

In [ ]:
# ── FIRMS를 FeatureCollection으로 로드 ────────────────────────
print('Loading FIRMS as FeatureCollection...')

# FIRMS는 포인트 벡터 데이터
firms_fc = (
    ee.FeatureCollection('FIRMS')
    .filterBounds(GANGWON)
    .filterDate(FIRMS_START, FIRMS_END)
)

count = firms_fc.size().getInfo()
print(f'Total fire points in Gangwon-do: {count:,}')

# 샘플 확인
sample = firms_fc.limit(3).getInfo()
print('\n[샘플 속성]')
for f in sample['features']:
    print(f'  {f["properties"]}')

In [ ]:
# ── 임상도 기반 수종 레이어 로드 ──────────────────────────────
modis_lc = (
    ee.ImageCollection('MODIS/061/MCD12Q1')
    .filterDate('2022-01-01', '2022-12-31')
    .first()
    .select('LC_Type1')
    .clip(GANGWON)
)

species_map = (
    ee.Image(3)
    .where(modis_lc.eq(1).Or(modis_lc.eq(3)), 0)   # 침엽수
    .where(modis_lc.eq(2).Or(modis_lc.eq(4)), 1)   # 활엽수
    .where(modis_lc.eq(5), 2)                        # 혼효림
    .rename('species')
    .clip(GANGWON)
)

# ── 수종별 산불 발생 빈도 계산 ────────────────────────────────
print('Calculating fire frequency by tree species...')

species_fire_stats = {}
fire_binary = fire_count.gt(0).rename('fire_binary')  # 화점 발생 여부

for cls_id, cls_name in SPECIES_CLASSES.items():
    cls_mask = species_map.eq(cls_id)

    # 해당 수종 면적 (픽셀 수)
    total_px = cls_mask.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=GANGWON, scale=500,
        maxPixels=1e12, bestEffort=True
    ).getInfo().get('species', 0)

    # 해당 수종 내 화점 픽셀 수
    # fire_binary 밴드명이 'fire_binary'로 변경됨
    fire_px = fire_binary.updateMask(cls_mask).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=GANGWON, scale=500,
        maxPixels=1e12, bestEffort=True
    ).getInfo().get('fire_binary', 0)

    fire_rate = fire_px / total_px * 100 if total_px > 0 else 0
    species_fire_stats[cls_name] = {
        'total_pixels': total_px,
        'fire_pixels':  fire_px,
        'fire_rate_%':  round(fire_rate, 3),
        'area_ha':      total_px * 500 * 500 / 10000,
    }
    print(f'  {cls_name:15s}: 면적={total_px*500*500/10000:8,.0f}ha  '
          f'화점={fire_px:6,}px  비율={fire_rate:.3f}%')

print('\nFire frequency by species calculated.')

In [ ]:
# ── 수종별 산불 통계 시각화 ───────────────────────────────────
df = pd.DataFrame(species_fire_stats).T.reset_index()
df.columns = ['Species', 'Total Pixels', 'Fire Pixels', 'Fire Rate (%)', 'Area (ha)']
df = df.sort_values('Fire Rate (%)', ascending=False)

print('\n[Species Fire Risk Statistics]')
print('=' * 65)
print(df[['Species','Area (ha)','Fire Pixels','Fire Rate (%)']].to_string(index=False))
print('=' * 65)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Fire Occurrence Frequency by Tree Species\nGangwon-do (FIRMS VIIRS)',
    fontsize=13, fontweight='bold'
)

colors = [SPECIES_COLORS[list(SPECIES_CLASSES.values()).index(s)]
          for s in df['Species']]

# 화점 비율 막대
bars = axes[0].bar(df['Species'], df['Fire Rate (%)'],
                   color=colors, edgecolor='k', linewidth=0.5)
for bar, val in zip(bars, df['Fire Rate (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f'{val:.3f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Fire Pixel Ratio (%)')
axes[0].set_title('Fire Occurrence Rate by Species')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# 면적 파이차트
axes[1].pie(
    df['Area (ha)'],
    labels=df['Species'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8
)
axes[1].set_title('Forest Area Distribution by Species')

plt.tight_layout()
plt.savefig('species_fire_stats.png', bbox_inches='tight', dpi=150)
plt.show()
print("'species_fire_stats.png' saved.")

## Section 3 — Sentinel-2 수종 분류 딥러닝

### 접근 방법
- **입력**: Sentinel-2 4계절 시계열 (봄/여름/가을/겨울) × 6밴드 = 24채널
- **레이블**: MODIS 토지피복 재분류 (4클래스)
- **모델**: U-Net (ResNet34 인코더)
- **전략**: 의사 레이블(MODIS) → U-Net 학습 → 고해상도(10m) 수종 지도 생성

```
Sentinel-2 (10m, 24ch)  →  U-Net  →  수종 분류 (4클래스, 10m)
MODIS LC (500m)         →  리샘플 → 학습 레이블
```

In [ ]:
# ── Sentinel-2 4계절 시계열 로드 ──────────────────────────────
print('Loading Sentinel-2 multi-season imagery...')

BANDS = ['B2','B3','B4','B8','B11','B12']  # Blue,Green,Red,NIR,SWIR1,SWIR2

def load_s2_season(start, end):
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(GANGWON)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .select(BANDS)
        .median()
        .clip(GANGWON)
    )

# 4계절 합성
s2_spring = load_s2_season('2022-03-01', '2022-05-31')  # 봄: 신록 시작
s2_summer = load_s2_season('2022-06-01', '2022-08-31')  # 여름: 최대 엽면적
s2_autumn = load_s2_season('2022-09-01', '2022-11-30')  # 가을: 단풍 (침/활 구분 최대)
s2_winter = load_s2_season('2022-12-01', '2023-02-28')  # 겨울: 낙엽 후 상록 구분

# 4계절 스택 (24채널)
s2_stack = (
    s2_spring.rename([f'SP_{b}' for b in BANDS])
    .addBands(s2_summer.rename([f'SU_{b}' for b in BANDS]))
    .addBands(s2_autumn.rename([f'AU_{b}' for b in BANDS]))
    .addBands(s2_winter.rename([f'WI_{b}' for b in BANDS]))
)

print(f'Input bands: {len(BANDS) * 4} channels (6 bands × 4 seasons)')
print('Seasons: Spring(Mar-May) / Summer(Jun-Aug) / Autumn(Sep-Nov) / Winter(Dec-Feb)')

# 가을 False Color 시각화 (침/활 구분 잘 됨)
print('\nDownloading autumn false color image...')
autumn_fc = get_thumb_image(s2_autumn, ['B8','B4','B3'], 0, 4000)
summer_fc = get_thumb_image(s2_summer, ['B8','B4','B3'], 0, 4000)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sentinel-2 False Color (NIR-Red-Green)\nGangwon-do 2022',
             fontsize=13, fontweight='bold')
axes[0].imshow(summer_fc)
axes[0].set_title('Summer (Jun–Aug)\nHealthy vegetation = bright red', fontsize=10)
axes[0].axis('off')
axes[1].imshow(autumn_fc)
axes[1].set_title('Autumn (Sep–Nov)\nConifers=dark red, Broadleaf=orange/yellow', fontsize=10)
axes[1].axis('off')
plt.tight_layout()
plt.savefig('s2_seasonal.png', bbox_inches='tight', dpi=150)
plt.show()
print("'s2_seasonal.png' saved.")

In [ ]:
# ── GeoTIFF 다운로드 ──────────────────────────────────────────
# 강원도 전체는 너무 크므로 대표 서브영역 사용 (학습/추론)
# 강릉·속초·양양 일대 (산불 다발 + 다양한 수종)
ROI_DL = ee.Geometry.Rectangle([128.3, 37.6, 129.0, 38.2])

os.makedirs('/content/data', exist_ok=True)
S2_FILE      = '/content/data/s2_stack.tif'
SPECIES_FILE = '/content/data/species_label.tif'
FIRE_FILE    = '/content/data/fire_count.tif'

def download_if_missing(image, filepath, region, scale, label):
    if os.path.exists(filepath):
        print(f'  Already exists: {filepath}')
    else:
        print(f'  Downloading: {label}...')
        geemap.download_ee_image(
            image=image, filename=filepath,
            region=region, scale=scale, crs='EPSG:4326'
        )
        print(f'  Saved: {filepath}')

print('Downloading GeoTIFF files...')
download_if_missing(s2_stack,    S2_FILE,      ROI_DL, 100, 'Sentinel-2 stack (24ch, 100m)')
download_if_missing(species_map, SPECIES_FILE, ROI_DL, 100, 'Species label (MODIS, 100m)')
download_if_missing(fire_count,  FIRE_FILE,    ROI_DL, 100, 'Fire count (FIRMS, 100m)')
print('\nAll files ready.')

In [ ]:
# ── GeoTIFF → NumPy ───────────────────────────────────────────
import rasterio

def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32), src.meta

s2_data,  _  = load_tif(S2_FILE)       # (24, H, W)
label_data, _ = load_tif(SPECIES_FILE)  # (1, H, W)
fire_data,  _ = load_tif(FIRE_FILE)    # (1, H, W)

H, W = s2_data.shape[1], s2_data.shape[2]

# 유효 픽셀 마스크
valid = (
    (s2_data[0] > 0) &
    np.isfinite(label_data[0]) &
    (label_data[0] >= 0) & (label_data[0] <= 3)
)

print(f'Image size  : {H} × {W}  ({H*W:,} pixels)')
print(f'Valid pixels: {valid.sum():,}  ({valid.mean()*100:.1f}%)')
print(f'Input bands : {s2_data.shape[0]} (6 bands × 4 seasons)')
print(f'\nLabel distribution:')
for cls_id, cls_name in SPECIES_CLASSES.items():
    cnt = ((label_data[0] == cls_id) & valid).sum()
    pct = cnt / valid.sum() * 100
    print(f'  {cls_name:15s}: {cnt:6,}px  ({pct:5.1f}%)')

In [ ]:
# ── Dataset & DataLoader ──────────────────────────────────────
PATCH_SIZE = 64    # 패치 크기
STRIDE     = 32    # 슬라이딩 윈도우 스트라이드
N_CLASSES  = 4
N_BANDS    = 24

class SpeciesDataset(Dataset):
    """
    Sentinel-2 24채널 패치 기반 수종 분류 Dataset
    슬라이딩 윈도우로 전체 이미지에서 패치 추출
    """
    def __init__(self, s2, label, valid_mask, patch=64, stride=32, augment=False):
        self.s2     = s2      # (24, H, W)
        self.label  = label   # (H, W)
        self.valid  = valid_mask
        self.patch  = patch
        self.augment = augment

        # 패치 위치 생성 (유효 픽셀 50% 이상인 패치만)
        self.coords = []
        for y in range(0, s2.shape[1] - patch, stride):
            for x in range(0, s2.shape[2] - patch, stride):
                patch_valid = valid_mask[y:y+patch, x:x+patch]
                if patch_valid.mean() > 0.5:
                    self.coords.append((y, x))

        print(f'  Dataset: {len(self.coords)} patches '
              f'(patch={patch}×{patch}, stride={stride})')

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        y, x = self.coords[idx]
        img = self.s2[:, y:y+self.patch, x:x+self.patch].copy()
        lbl = self.label[y:y+self.patch, x:x+self.patch].copy()

        # 정규화: DN → 반사율 (÷10000)
        img = img / 10000.0
        img = np.clip(img, 0, 1)

        # nodata 처리
        lbl = np.where(np.isfinite(lbl), lbl, 3).astype(np.int64)
        lbl = np.clip(lbl, 0, 3)

        # 데이터 증강 (좌우/상하 반전)
        if self.augment and np.random.rand() > 0.5:
            img = img[:, :, ::-1].copy()
            lbl = lbl[:, ::-1].copy()
        if self.augment and np.random.rand() > 0.5:
            img = img[:, ::-1, :].copy()
            lbl = lbl[::-1, :].copy()

        return torch.FloatTensor(img), torch.LongTensor(lbl)


label_2d = label_data[0].astype(np.float32)

print('Building dataset...')
dataset = SpeciesDataset(s2_data, label_2d, valid,
                         patch=PATCH_SIZE, stride=STRIDE, augment=True)

# 학습/검증 분할 (8:2)
n_val   = int(len(dataset) * 0.2)
n_train = len(dataset) - n_val
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)

print(f'Train: {n_train} patches  |  Val: {n_val} patches')

In [ ]:
# ── U-Net 모델 정의 ───────────────────────────────────────────
print('Building U-Net model...')

# 24채널 입력을 처리하기 위해 첫 번째 Conv 레이어 수정
unet = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,    # 24ch 입력이라 ImageNet 가중치 직접 사용 불가
    in_channels=N_BANDS,     # 24채널 (6밴드 × 4계절)
    classes=N_CLASSES,       # 4클래스
    activation=None,         # CrossEntropy loss에서 softmax 내장
).to(device)

total_params   = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# 클래스 가중치 (불균형 보정: 비산림이 많으면 가중치 낮춤)
cls_counts = np.array([
    ((label_2d == i) & valid).sum() for i in range(N_CLASSES)
], dtype=np.float32)
cls_weights = 1.0 / (cls_counts + 1e-6)
cls_weights /= cls_weights.sum()
cls_weights_t = torch.FloatTensor(cls_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=cls_weights_t, ignore_index=-1)
optimizer = optim.AdamW(unet.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print(f'Class weights: {dict(zip(SPECIES_CLASSES.values(), cls_weights.round(3)))}')

In [ ]:
# ── U-Net 학습 ────────────────────────────────────────────────
EPOCHS = 30

def compute_miou(pred, target, n_classes=4):
    """mean IoU 계산"""
    ious = []
    pred_flat   = pred.cpu().numpy().flatten()
    target_flat = target.cpu().numpy().flatten()
    for c in range(n_classes):
        tp = ((pred_flat == c) & (target_flat == c)).sum()
        fp = ((pred_flat == c) & (target_flat != c)).sum()
        fn = ((pred_flat != c) & (target_flat == c)).sum()
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return np.mean(ious) if ious else 0.0


train_losses, val_losses, val_mious = [], [], []

print(f'Training U-Net  |  Epochs={EPOCHS}  |  Device={device}')
print('-' * 60)

for epoch in range(EPOCHS):
    # ── 학습 ──
    unet.train()
    ep_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = unet(imgs)                  # (B, 4, H, W)
        loss   = criterion(logits, lbls)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    avg_train = ep_loss / len(train_loader)

    # ── 검증 ──
    unet.eval()
    val_loss, all_preds, all_lbls = 0.0, [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = unet(imgs)
            val_loss += criterion(logits, lbls).item()
            all_preds.append(logits.argmax(dim=1))
            all_lbls.append(lbls)
    avg_val  = val_loss / len(val_loader)
    miou     = compute_miou(
        torch.cat(all_preds), torch.cat(all_lbls)
    )

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    val_mious.append(miou)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1:2d}/{EPOCHS}]  '
              f'Train Loss: {avg_train:.4f}  '
              f'Val Loss: {avg_val:.4f}  '
              f'mIoU: {miou:.3f}')

print('-' * 60)
print(f'Best mIoU: {max(val_mious):.3f}  (Epoch {val_mious.index(max(val_mious))+1})')

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses,   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training / Validation Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(val_mious, color='green')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Validation mIoU')
axes[1].grid(True, alpha=0.3)

plt.suptitle('U-Net Training Curves — Tree Species Classification', fontsize=13)
plt.tight_layout()
plt.savefig('unet_training.png', bbox_inches='tight', dpi=150)
plt.show()
print("'unet_training.png' saved.")

In [ ]:
# ── 전체 영역 추론 (경계 artifact 수정) ──────────────────────
print('Running inference on full area...')

unet.eval()
pred_map = np.full((H, W), -1, dtype=np.int32)

with torch.no_grad():
    for y in range(0, H, PATCH_SIZE):
        for x in range(0, W, PATCH_SIZE):
            # 실제 패치 범위 (이미지 경계 초과 방지)
            y_end = min(y + PATCH_SIZE, H)
            x_end = min(x + PATCH_SIZE, W)
            ph = y_end - y
            pw = x_end - x

            patch = s2_data[:, y:y_end, x:x_end] / 10000.0
            patch = np.clip(patch, 0, 1)

            # 패치가 PATCH_SIZE보다 작으면 패딩
            if ph < PATCH_SIZE or pw < PATCH_SIZE:
                padded = np.zeros((N_BANDS, PATCH_SIZE, PATCH_SIZE),
                                  dtype=np.float32)
                padded[:, :ph, :pw] = patch
                patch = padded

            t = torch.FloatTensor(patch).unsqueeze(0).to(device)
            logits = unet(t)
            pred   = logits.argmax(dim=1).squeeze(0).cpu().numpy()

            # 패딩 제거 후 실제 크기만 저장
            pred_map[y:y_end, x:x_end] = pred[:ph, :pw]

# nodata 마스크 적용
pred_map[~valid] = -1

# 결과 통계
print('\n[Species Classification Result]')
print('=' * 45)
for cls_id, cls_name in SPECIES_CLASSES.items():
    cnt = (pred_map == cls_id).sum()
    ha  = cnt * 100 * 100 / 10000
    bar = '█' * int(cnt / valid.sum() * 40)
    print(f'  {cls_name:15s}: {ha:8,.0f} ha  {bar}')
print('=' * 45)

In [ ]:
# ── 수종 분류 결과 시각화 ─────────────────────────────────────
def pred_to_rgb(pred_map, colors, valid_mask):
    rgb = np.ones((*pred_map.shape, 3), dtype=np.float32)
    for cls_id, color in enumerate(colors):
        r, g, b = mcolors.to_rgb(color)
        mask = (pred_map == cls_id) & valid_mask
        rgb[mask] = [r, g, b]
    rgb[~valid_mask] = [0.9, 0.9, 0.9]  # nodata = 연회색
    return rgb


label_rgb = pred_to_rgb(label_2d.astype(int), SPECIES_COLORS, valid)
pred_rgb  = pred_to_rgb(pred_map, SPECIES_COLORS, valid)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(
    'Tree Species Classification — U-Net Result\n'
    'Gangwon-do (Gangneung–Sokcho–Yangyang area)',
    fontsize=13, fontweight='bold'
)

# 가을 RGB
autumn_rgb_np = np.zeros((H, W, 3), dtype=np.float32)
for i, ch in enumerate([4, 3, 2]):  # B4, B3, B2 (계절 인덱스: 가을=12~17)
    band = s2_data[12 + ch] / 3000.0
    autumn_rgb_np[:,:,i] = np.clip(band, 0, 1)
autumn_rgb_np[~valid] = 0.9

axes[0].imshow(autumn_rgb_np)
axes[0].set_title('Sentinel-2 Autumn RGB', fontsize=11)
axes[0].axis('off')

axes[1].imshow(label_rgb)
axes[1].set_title('MODIS Label (500m, resampled)', fontsize=11)
axes[1].axis('off')

axes[2].imshow(pred_rgb)
axes[2].set_title(f'U-Net Prediction (mIoU={max(val_mious):.3f})', fontsize=11)
axes[2].axis('off')

legend = [
    mpatches.Patch(color=c, label=l)
    for c, l in zip(SPECIES_COLORS, SPECIES_CLASSES.values())
]
fig.legend(handles=legend, loc='lower center', ncol=4,
           fontsize=11, bbox_to_anchor=(0.5, -0.04),
           markerscale=2.0, framealpha=0.9)

plt.tight_layout()
plt.savefig('species_classification.png', bbox_inches='tight', dpi=150)
plt.show()
print("'species_classification.png' saved.")

## Section 4 — 산불 위험도 지도 생성

### 위험도 계산 공식

$$\text{Fire Risk} = w_1 \cdot R_{species} + w_2 \cdot R_{slope} + w_3 \cdot R_{dryness} + w_4 \cdot R_{history}$$

| 인자 | 가중치 | 설명 |
|------|--------|------|
| **수종 위험계수** | 0.35 | 침엽수=1.0, 혼효=0.6, 활엽=0.25, 비산림=0.05 |
| **경사도** | 0.25 | 경사가 급할수록 산불 확산 빠름 |
| **건조도 (NDMI)** | 0.25 | 식생 수분 부족할수록 인화성 높음 |
| **산불 이력** | 0.15 | 과거 화점 밀도 높을수록 위험 |


In [ ]:
# ── 위험도 인자 계산 ──────────────────────────────────────────

# 1. 수종 위험계수 맵
risk_coef = {0: 1.00, 1: 0.25, 2: 0.60, 3: 0.05}  # cls_id: risk
species_risk_map = np.zeros((H, W), dtype=np.float32)
for cls_id, risk in risk_coef.items():
    species_risk_map[pred_map == cls_id] = risk
species_risk_map[~valid] = 0.0

# 2. 경사도 (DEM에서 계산)
dem = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(ROI_DL)
    .mosaic()
    .select('DEM')
    .clip(ROI_DL)
)
slope = ee.Terrain.slope(dem).clip(ROI_DL)  # 단위: 도(degree)

SLOPE_FILE = '/content/data/slope.tif'
download_if_missing(slope, SLOPE_FILE, ROI_DL, 100, 'Slope (degree)')
slope_data, _ = load_tif(SLOPE_FILE)
slope_norm = np.clip(slope_data[0] / 45.0, 0, 1)  # 0~45도 → 0~1 정규화
slope_norm[~valid] = 0.0

# 3. 건조도 (NDMI: Normalized Difference Moisture Index)
# NDMI = (NIR - SWIR1) / (NIR + SWIR1): 낮을수록 건조
# 여름철 건조도가 산불 위험과 가장 상관높음
ndmi_summer = s2_summer.normalizedDifference(['B8', 'B11']).clip(ROI_DL)
NDMI_FILE = '/content/data/ndmi_summer.tif'
download_if_missing(ndmi_summer, NDMI_FILE, ROI_DL, 100, 'NDMI Summer')
ndmi_data, _ = load_tif(NDMI_FILE)
# NDMI 낮을수록(건조) 위험 높음 → 반전 후 정규화
dryness = np.clip((-ndmi_data[0] + 0.5) / 1.0, 0, 1)
dryness[~valid] = 0.0

# 4. 산불 이력 밀도
fire_norm = np.clip(fire_data[0] / 10.0, 0, 1)  # 최대 10회 기준 정규화
fire_norm[~valid] = 0.0

print('Risk factors calculated:')
print(f'  Species risk  — mean: {species_risk_map[valid].mean():.3f}')
print(f'  Slope         — mean: {slope_norm[valid].mean():.3f}')
print(f'  Dryness (NDMI)— mean: {dryness[valid].mean():.3f}')
print(f'  Fire history  — mean: {fire_norm[valid].mean():.3f}')

In [ ]:
# ── 종합 위험도 계산 ──────────────────────────────────────────

# 가중치 설정
W_SPECIES = 0.35
W_SLOPE   = 0.25
W_DRYNESS = 0.25
W_HISTORY = 0.15

fire_risk = (
    W_SPECIES * species_risk_map +
    W_SLOPE   * slope_norm +
    W_DRYNESS * dryness +
    W_HISTORY * fire_norm
)
fire_risk[~valid] = np.nan

# 5등급 위험도 분류
RISK_BINS   = [0.0, 0.2, 0.35, 0.50, 0.65, 1.01]
RISK_LABELS = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
RISK_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

risk_cls = np.full((H, W), -1, dtype=np.int32)
for i, (lo, hi) in enumerate(zip(RISK_BINS[:-1], RISK_BINS[1:])):
    mask_i = (fire_risk >= lo) & (fire_risk < hi) & valid
    risk_cls[mask_i] = i

# 등급별 면적 통계
print('[Fire Risk Area Summary]')
print('=' * 50)
for i, (label, color) in enumerate(zip(RISK_LABELS, RISK_COLORS)):
    cnt = (risk_cls == i).sum()
    ha  = cnt * 100 * 100 / 10000
    pct = cnt / valid.sum() * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:12s}: {ha:8,.0f} ha  ({pct:4.1f}%)  {bar}')
print('=' * 50)
print(f'  Mean risk score: {np.nanmean(fire_risk):.3f}')
print(f'  Max  risk score: {np.nanmax(fire_risk):.3f}')

In [ ]:
# ── DEM & Slope 재계산 ────────────────────────────────────────
import os

# 기존 파일 삭제 후 재다운로드
if os.path.exists(SLOPE_FILE):
    os.remove(SLOPE_FILE)

# Copernicus DEM 대신 SRTM 사용 (더 안정적)
dem = ee.Image('USGS/SRTMGL1_003').clip(ROI_DL)

# nodata 처리 후 경사도 계산
slope = ee.Terrain.slope(dem).clip(ROI_DL)

# 유효값 확인
slope_stats = slope.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        ee.Reducer.mean(), sharedInputs=True
    ),
    geometry=ROI_DL, scale=100,
    maxPixels=1e10, bestEffort=True
).getInfo()
print('[Slope stats from GEE]')
print(slope_stats)

# 다운로드
print('\nDownloading slope...')
geemap.download_ee_image(
    image=slope,
    filename=SLOPE_FILE,
    region=ROI_DL,
    scale=100,
    crs='EPSG:4326'
)

# 재로드 및 확인
slope_data, _ = load_tif(SLOPE_FILE)
print(f'\nSlope reloaded:')
print(f'  Shape: {slope_data.shape}')

# -inf, nan 처리
slope_arr = slope_data[0].copy()
slope_arr = np.where(np.isfinite(slope_arr), slope_arr, 0.0)
print(f'  Min : {slope_arr.min():.2f}°')
print(f'  Max : {slope_arr.max():.2f}°')
print(f'  Mean: {slope_arr[valid].mean():.2f}°')

# 정규화
slope_norm = np.clip(slope_arr / 45.0, 0, 1)
slope_norm[~valid] = 0.0
print(f'  Normalized mean: {slope_norm[valid].mean():.4f}')

# 시각화
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(slope_norm, cmap='OrRd', vmin=0, vmax=1)
ax.set_title('Slope Risk Factor (SRTM)', fontsize=12)
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
             label='Normalized slope (0=flat, 1=steep)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 위험도 지도 최종 시각화 ───────────────────────────────────

cmap_risk = ListedColormap(RISK_COLORS)
norm_risk = BoundaryNorm(range(6), cmap_risk.N)

fig = plt.figure(figsize=(22, 16))
fig.suptitle(
    'Wildfire Risk Assessment Map\n'
    'Gangwon-do (Gangneung–Sokcho–Yangyang)',
    fontsize=15, fontweight='bold'
)

gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.15)

# ── 패널 1: 수종 분류 결과 ────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(pred_rgb)
ax1.set_title('① Tree Species Classification\n(U-Net)', fontsize=11)
ax1.axis('off')
leg1 = [mpatches.Patch(color=c, label=l)
        for c, l in zip(SPECIES_COLORS, SPECIES_CLASSES.values())]
ax1.legend(handles=leg1, loc='lower left', fontsize=8, framealpha=0.9)

# ── 패널 2: 경사도 ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
slope_vis = slope_norm.copy()
slope_vis[~valid] = np.nan
im2 = ax2.imshow(slope_vis, cmap='OrRd', vmin=0, vmax=1)
ax2.set_title('② Slope Risk Factor\n(steeper = higher risk)', fontsize=11)
ax2.axis('off')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label='Normalized slope')

# ── 패널 3: 건조도 ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
dry_vis = dryness.copy()
dry_vis[~valid] = np.nan
im3 = ax3.imshow(dry_vis, cmap='YlOrBr', vmin=0, vmax=1)
ax3.set_title('③ Dryness Risk Factor\n(NDMI-based, drier = higher risk)', fontsize=11)
ax3.axis('off')
plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label='Dryness (0=wet, 1=dry)')

# ── 패널 4: 산불 이력 ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
fire_vis = fire_norm.copy()
fire_vis[~valid] = np.nan
im4 = ax4.imshow(fire_vis, cmap='Reds', vmin=0, vmax=1)
ax4.set_title('④ Fire History Factor\n(FIRMS, higher = more frequent)', fontsize=11)
ax4.axis('off')
plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04, label='Normalized fire count')

# ── 패널 5: 종합 위험도 연속값 ────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
im5 = ax5.imshow(fire_risk, cmap='RdYlGn_r', vmin=0, vmax=1)
ax5.set_title(
    f'⑤ Composite Risk Score\n'
    f'(S:{W_SPECIES} + Sl:{W_SLOPE} + D:{W_DRYNESS} + H:{W_HISTORY})',
    fontsize=11
)
ax5.axis('off')
plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04, label='Risk score (0=low, 1=high)')

# ── 패널 6: 5등급 위험도 지도 (메인) ─────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
risk_vis = risk_cls.astype(float)
risk_vis[~valid] = np.nan
ax6.imshow(risk_vis, cmap=cmap_risk, norm=norm_risk, interpolation='nearest')
ax6.set_title('⑥ Fire Risk Class Map\n(5-level classification)', fontsize=11)
ax6.axis('off')

# 위험도 면적 범례
leg6 = [
    mpatches.Patch(
        color=c,
        label=f'{l}: {(risk_cls==i).sum()*100*100/10000:,.0f} ha'
    )
    for i, (c, l) in enumerate(zip(RISK_COLORS, RISK_LABELS))
]
ax6.legend(handles=leg6, loc='lower left', fontsize=8,
           framealpha=0.95, title='Risk Class', title_fontsize=9)

plt.savefig('wildfire_risk_map.png', bbox_inches='tight', dpi=150)
plt.show()
print("'wildfire_risk_map.png' saved.")

## Section 5 — 최종 요약 보고서

In [ ]:
print('=' * 65)
print('  Gangwon-do Wildfire Risk Assessment — Summary Report')
print('=' * 65)

print(f'\n[Analysis Area]')
print(f'  Region      : Gangwon-do (Gangneung–Sokcho–Yangyang)')
print(f'  Valid pixels: {valid.sum():,}  (100m resolution)')
print(f'  Total area  : {valid.sum() * 100 * 100 / 10000:,.0f} ha')

print(f'\n[Fire History (FIRMS VIIRS {FIRMS_START[:4]}–{FIRMS_END[:4]})]')
for k, v in species_fire_stats.items():
    print(f'  {k:15s}: {v["fire_rate_%"]:.3f}% fire rate')

print(f'\n[Species Classification (U-Net)]')
print(f'  Best mIoU   : {max(val_mious):.3f}')
for cls_id, cls_name in SPECIES_CLASSES.items():
    cnt = (pred_map == cls_id).sum()
    ha  = cnt * 100 * 100 / 10000
    print(f'  {cls_name:15s}: {ha:8,.0f} ha  '
          f'(Risk coef={risk_coef[cls_id]:.2f})')

print(f'\n[Fire Risk Assessment]')
print(f'  Weights: Species={W_SPECIES} / '
      f'Slope={W_SLOPE} / Dryness={W_DRYNESS} / History={W_HISTORY}')
for i, label in enumerate(RISK_LABELS):
    cnt = (risk_cls == i).sum()
    ha  = cnt * 100 * 100 / 10000
    pct = cnt / valid.sum() * 100
    print(f'  {label:12s}: {ha:8,.0f} ha  ({pct:.1f}%)')

high_risk_ha = sum(
    (risk_cls == i).sum() * 100 * 100 / 10000
    for i, l in enumerate(RISK_LABELS)
    if l in ['High', 'Very High']
)
print(f'\n  ⚠️  High + Very High risk area: {high_risk_ha:,.0f} ha')
print('=' * 65)

---
## 한계점 & 개선 방향

| 한계 | 개선 방법 |
|------|----------|
| MODIS 레이블이 500m로 거칠음 | 산림청 임상도(25m) SHP 업로드 후 대체 |
| 수종 분류 4클래스만 | 임상도 기준 소나무/잣나무/낙엽송 세분화 |
| 건조도가 여름 단일 시점 | 계절별 NDMI 최솟값 시계열 사용 |
| 바람 방향 미반영 | ERA5 바람 데이터 추가 |
| 위험도 가중치 문헌값 | 실제 산불 데이터로 회귀 보정 |

### 임상도 직접 사용 방법
```python
# 산림청 산림공간정보서비스에서 임상도 SHP 다운로드 후
# Google Drive에 업로드 → GEE에 Asset으로 등록
# https://map.forest.go.kr

forest_map = ee.FeatureCollection('users/YOUR_ID/gangwon_imSangDo')
species_label = forest_map.reduceToImage(
    properties=['FMST_CL_CD'],  # 임상 코드
    reducer=ee.Reducer.first()
)
```